# 🇺🇿 nanoGPT-uz — nanoGPT + Llama + DeepSeek, bitta notebookda (JAX / TPU)

Andrej Karpathy'ning [nanoGPT](https://github.com/karpathy/nanoGPT) g'oyasi — **sodda, o'qiladigan, noldan o'qitiladigan GPT** — saqlangan. Uning ustiga zamonaviy modellardan eng samarali qismlar qo'shilgan. Hammasi **JAX** da yozilgan, chunki TPU uchun JAX eng to'g'ri va tez yo'l (XLA kompilyatsiya, bf16, `jit`).

| Qism | Manba | nanoGPT (asl) | nanoGPT-uz |
|---|---|---|---|
| Pozitsiya | Llama | o'rganiladigan absolyut embedding | **RoPE** |
| Normalizatsiya | Llama | LayerNorm | **RMSNorm** (pre-norm) |
| MLP | Llama | GELU | **SwiGLU** |
| Attention | DeepSeek-V2/V3 | oddiy MHA | **MLA** (Multi-head Latent Attention: KV past o'lchamli latentga siqiladi, RoPE ajratilgan) — yoki Llama **GQA** |
| FFN | DeepSeekMoE | dense | **MoE**: 1 ta umumiy (shared) + 16 ta mayda (fine-grained) ekspert, top-2 |
| Ekspert balansi | DeepSeek-V3 | — | **aux-loss-free** (bias bilan) + juda kichik balans loss |
| Qo'shimcha maqsad | DeepSeek-V3 | — | **MTP** (multi-token prediction, ixtiyoriy) |
| Optimizer | modded-nanoGPT / Moonlight | AdamW | **Muon** (yashirin matritsalar) + AdamW (embedding, head, norm) |
| Barqarorlik | Gemma 2 / modded-nanoGPT | — | logit soft-capping, latent/QK-norm, chiqish proyeksiyalari 0 dan |
| Attention mask | — | hujjatlar aralashadi | **hujjat ichida** (doc-masking) |
| LR jadvali | MiniCPM | cosine | **WSD** (vaqtga bog'liq — Colab uzilsa ham to'g'ri davom etadi) |
| Loss | — | to'liq logits | **bo'laklab** (chunked) cross-entropy — xotira tejaydi |
| Tokenizer | — | GPT-2 BPE (inglizcha) | **o'zbekcha BPE 32k** (o'zbek, g'alaba, ma'no — bitta bo'lak) |
| Bosqichlar | — | faqat pretrain | **pretrain → SFT (chat)** |

### Colab TPU v5e-1 (bepul) uchun
1. `Runtime → Change runtime type → TPU v5e-1`.
2. Avval `PROFILE='smoke'` bilan 3–5 daqiqa sinang (tiny model, oz data), keyin `PROFILE='real'`.
3. Google Drive ulansa, data va checkpointlar Drive'da saqlanadi — Colab uzilsa, **Run all** qilib davom ettirasiz (oxirgi checkpointdan).

**Halol kutilma:** bitta bepul TPU'da ~2 soatlik trening (~0.4–0.5 milliard token) — o'zbekcha ravon matn yozadigan va oddiy savollarga javob beradigan kichik model. Faktlarda xato qiladi; katta modellar darajasi emas. Natijani val loss va namunalar bilan baholang.

In [ ]:
# 0. Muhit: TPU, kutubxonalar, Google Drive
import os, sys, re, json, math, time, hashlib, shutil, subprocess, importlib.util, warnings, functools
from pathlib import Path
from collections import Counter
from itertools import islice
import numpy as np

T0 = time.time()
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=SyntaxWarning)
for pkg in ('tokenizers', 'datasets', 'optax'):
    if importlib.util.find_spec(pkg) is None:
        print(f'⚠️ {pkg} o\'rnatilmoqda...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

import jax, jax.numpy as jnp, optax
import tokenizers, datasets
print('JAX', jax.__version__, '| optax', optax.__version__, '| tokenizers', tokenizers.__version__, '| datasets', datasets.__version__)
DEVICES = jax.devices()
PLATFORM = DEVICES[0].platform
print('Qurilmalar:', DEVICES)
if PLATFORM != 'tpu':
    print('⚠️ TPU topilmadi. Colab: Runtime → Change runtime type → TPU v5e-1. (smoke rejim CPU/GPU da ham ishlaydi)')

# TPU avlodiga qarab bf16 cho'qqi quvvati (MFU hisoblash uchun)
_kind = getattr(DEVICES[0], 'device_kind', '').lower()
PEAK_FLOPS = next((v for k, v in [('v6', 918e12), ('v5 lite', 197e12), ('v5e', 197e12), ('v5p', 459e12), ('v5', 459e12),
                                   ('v4', 275e12), ('v3', 123e12), ('v2', 45e12), ('t4', 65e12), ('a100', 312e12),
                                   ('l4', 121e12), ('h100', 989e12)] if k in _kind), None)
print('Qurilma turi:', _kind or PLATFORM, '| bf16 cho\'qqi:', f'{PEAK_FLOPS / 1e12:.0f} TFLOPS' if PEAK_FLOPS else 'noma\'lum')

USE_DRIVE = True   # Colab'da data/checkpoint Drive'da saqlansin (uzilishdan himoya)
DRIVE_OK = False
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_OK = True
    except Exception as e:  # Colab emas yoki rad etildi
        print('Drive ulanmadi:', repr(e)[:120])

## 1. Sozlamalar — faqat shu katakni o'zgartiring
- `PROFILE='smoke'` — tiny model va oz data bilan 3–5 daqiqalik to'liq sinov. Avval shuni ishlating.
- **Model sozlamalari** (`D_MODEL`, `N_LAYER`, `ATTN`, `N_EXPERTS`, `SEQ`, `VOCAB`...) checkpoint bilan birga saqlanadi. Ularni o'zgartirsangiz, yangi `WORKDIR` kerak (notebook buni tekshiradi).
- `PRETRAIN_MINUTES` / `SFT_MINUTES` — sof trening vaqti. Colab uzilsa, **Run all** — trening oxirgi checkpointdan davom etadi va vaqt hisobi saqlanadi.

In [ ]:
# =========== FAQAT SHU KATAKNI O'ZGARTIRING ===========
PROFILE = 'real'                 # 'real' | 'smoke' (3-5 daqiqalik to'liq sinov)

# --- Data (MAX_CHARS ≈ belgilar; 1e9 ≈ 1 GB matn ≈ 250M token) ---
USE_FW2 = True;   FW2_MAX_CHARS = 1.2e9    # HuggingFaceFW/fineweb-2 uzn_Latn — tozalangan o'zbekcha veb
USE_WIKI = True;  WIKI_MAX_CHARS = 3e8     # wikimedia/wikipedia 20231101.uz
USE_UZ_LM = True                           # ulugbek2013/uzbek-lang-lm — chat/instruction (pretrain + SFT)
USE_CODE = True;  CODE_MAX_CHARS = 2e8     # JetBrains-Research/OpenCodeInstruct-Clean — Python (mantiq uchun foydali)
VAL_PCT = 0.5
VOCAB = 32768
TOKENIZER_SAMPLE_CHARS = 1.5e8

# --- Model (nanoGPT + Llama + DeepSeek) ---
D_MODEL = 768; N_LAYER = 12; N_HEAD = 12
ATTN = 'mla'                     # 'mla' (DeepSeek) | 'gqa' (Llama)
MLA_KV_RANK = 256; MLA_QK_NOPE = 64; MLA_QK_ROPE = 32; MLA_V_DIM = 64
GQA_KV_HEADS = 4                 # faqat ATTN='gqa' uchun
N_DENSE_LAYERS = 1; DENSE_HIDDEN = 2048          # DeepSeek kabi: birinchi qatlam(lar) dense
N_EXPERTS = 16; TOP_K = 2; N_SHARED = 1; EXPERT_HIDDEN = 512   # DeepSeekMoE: fine-grained + shared
CAPACITY_FACTOR = 1.25           # treningda ekspert sig'imi (ortiqcha tokenlar shared ekspert orqali o'tadi)
BIAS_UPDATE = 1e-3               # aux-loss-free balans tezligi (DeepSeek-V3: γ)
BALANCE_LOSS = 1e-4              # juda kichik qo'shimcha balans loss (DeepSeek-V3: α)
MTP_WEIGHT = 0.0                 # DeepSeek MTP: 0.1-0.3 tajriba uchun (kichik modelda foydasi kafolatlanmagan)
SEQ = 1024; ROPE_THETA = 10000.0; SOFTCAP = 30.0
REMAT = True                     # qatlam aktivatsiyalarini qayta hisoblash (xotira tejaydi, katta batch)

# --- Trening ---
MICRO_BATCH = 16; ACCUM = 8      # 16 x 8 x 1024 = 131k token/qadam (sig'masa avtomatik moslashadi)
MUON_LR = 0.02; MUON_MOMENTUM = 0.95
ADAM_LR = 3e-3; ADAM_BETAS = (0.9, 0.95)
WEIGHT_DECAY = 0.0; GRAD_CLIP = 1.0
WARMUP_STEPS = 50; DECAY_FRAC = 0.3; MIN_LR_RATIO = 0.0   # WSD: oxirgi 30% vaqtda LR 0 gacha tushadi
PRETRAIN_MINUTES = 110           # sof trening vaqti (Colab uzilsa checkpointdan davom etadi)
SFT_MINUTES = 20; SFT_LR_SCALE = 0.25; SFT_MICRO_BATCH = 16; SFT_ACCUM = 4
EVAL_EVERY_MIN = 10; EVAL_BATCHES = 16; SAVE_EVERY_MIN = 20; LOG_EVERY = 20
SEED = 1337

WORKDIR = '/content/drive/MyDrive/nanogpt-uz' if DRIVE_OK else str(Path.cwd() / 'nanogpt-uz')
LOCAL_TMP = '/tmp/nanogpt_uz_raw'   # xom matn (vaqtinchalik)

SMOKE_ROWS = None
if PROFILE == 'smoke':
    D_MODEL = 128; N_LAYER = 3; N_HEAD = 4; MLA_KV_RANK = 64; MLA_QK_NOPE = 16; MLA_QK_ROPE = 16; MLA_V_DIM = 16
    DENSE_HIDDEN = 256; N_EXPERTS = 4; TOP_K = 2; EXPERT_HIDDEN = 64; GQA_KV_HEADS = 2
    VOCAB = 1024; SEQ = 128; MICRO_BATCH = 4; ACCUM = 2; SFT_MICRO_BATCH = 4; SFT_ACCUM = 1
    FW2_MAX_CHARS = WIKI_MAX_CHARS = CODE_MAX_CHARS = 3e5; SMOKE_ROWS = 300; VAL_PCT = 10
    TOKENIZER_SAMPLE_CHARS = 2e6; PRETRAIN_MINUTES = 1.0; SFT_MINUTES = 0.5; WARMUP_STEPS = 5
    EVAL_EVERY_MIN = 0.4; EVAL_BATCHES = 2; SAVE_EVERY_MIN = 0.5; LOG_EVERY = 5
    WORKDIR = str(Path(WORKDIR).parent / 'nanogpt-uz-smoke')

assert ATTN in ('mla', 'gqa') and PROFILE in ('real', 'smoke')
assert D_MODEL % N_HEAD == 0 and 1 <= TOP_K <= N_EXPERTS and VOCAB <= 65535
WORK = Path(WORKDIR); WORK.mkdir(parents=True, exist_ok=True)
DATA = WORK / 'data'; CKPT = WORK / 'checkpoints'; DATA.mkdir(exist_ok=True); CKPT.mkdir(exist_ok=True)
print('WORKDIR:', WORK, '| PROFILE:', PROFILE, '| ATTN:', ATTN, '| MoE:', f'{N_EXPERTS}x{EXPERT_HIDDEN} top-{TOP_K} + {N_SHARED} shared')

## 2. Data va o'zbekcha tokenizer
Manbalar: **FineWeb-2 `uzn_Latn`** (tozalangan o'zbekcha veb), **o'zbekcha Vikipediya**, **uzbek-lang-lm** (110k chat — pretrain va SFT uchun), ixtiyoriy **OpenCodeInstruct-Clean** (Python; har masala guruhidan bittadan — takrorlar olib tashlangan). Hammasi streaming orqali olinadi, dedup qilinadi, email/telefon maskalanadi, maxfiy kalitli hujjatlar tashlanadi. Tokenizer: 32k byte-level BPE (Rust), o'zbekcha apostrofli so'zlarni bo'lmaydi. Natija `WORKDIR/data` ga **uint16** ko'rinishida yoziladi va keyingi safar qayta yuklanmaydi.

In [ ]:
# 1. Data: yig'ish -> o'zbekcha BPE tokenizer -> uint16 token fayllari (bir marta; keyin Drive'dan olinadi)
from tokenizers import Regex, Tokenizer, decoders, models, pre_tokenizers, trainers

SPECIAL = ['<|pad|>', '<|bos|>', '<|eos|>', '<|user|>', '<|assistant|>', '<|system|>']
PRETOK = (r"[^\r\n\p{L}\p{N}]?\p{L}+(?:['ʻʼ‘’]\p{L}+)*|\p{N}{1,3}| ?[^\s\p{L}\p{N}]+[\r\n]*|\s*[\r\n]+|\s+(?!\S)|\s+")
_SPECIAL_RE = re.compile('|'.join(map(re.escape, SPECIAL)))
_SECRET = re.compile(r'-----BEGIN (?:RSA |OPENSSH |EC )?PRIVATE KEY-----|ghp_[A-Za-z0-9]{30,}|sk-[A-Za-z0-9_-]{32,}|AKIA[0-9A-Z]{16}|hf_[A-Za-z0-9]{30,}')
_EMAIL = re.compile(r'(?<![\w.+-])[\w.+-]+@[\w-]+(?:\.[\w-]+)*\.[A-Za-z]{2,}(?!\w)')
_PHONE = re.compile(r'(?<!\d)\+?998[\s-]?\(?\d{2}\)?[\s-]?\d{3}[\s-]?\d{2}[\s-]?\d{2}(?!\d)')

class UzTokenizer:
    """Byte-level BPE: har qanday matn yo'qotishsiz; o'zbekcha apostrofli so'zlar bo'linmaydi."""
    def __init__(self, tk):
        self.tk = tk; self.vocab_size = tk.get_vocab_size()
        self.pad_id, self.bos_id, self.eos_id, self.user_id, self.assistant_id, self.system_id = (tk.token_to_id(t) for t in SPECIAL)
    @classmethod
    def train(cls, texts, vocab_size):
        tk = Tokenizer(models.BPE())
        tk.pre_tokenizer = pre_tokenizers.Sequence([pre_tokenizers.Split(Regex(PRETOK), behavior='isolated'),
                                                    pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False)])
        tk.decoder = decoders.ByteLevel()
        tk.train_from_iterator(texts, trainers.BpeTrainer(vocab_size=vocab_size, min_frequency=2, special_tokens=SPECIAL,
                               initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False))
        return cls(tk)
    @classmethod
    def load(cls, path):
        return cls(Tokenizer.from_file(str(path)))
    def encode(self, s): return self.tk.encode(s, add_special_tokens=False).ids
    def encode_batch(self, xs): return [e.ids for e in self.tk.encode_batch(xs, add_special_tokens=False)]
    def decode(self, ids): return self.tk.decode([int(i) for i in ids], skip_special_tokens=True)

def chat_ids(tok, turns, add_generation_prompt=False):
    """<|bos|><|user|>\n savol \n<|assistant|>\n javob <|eos|> ...  -> (ids, loss_mask: faqat javoblarda 1)"""
    ids, mask = [tok.bos_id], [0]
    for role, text in turns:
        text = (text or '').strip()
        if role == 'assistant':
            head, body = [tok.assistant_id] + tok.encode('\n'), tok.encode(text) + [tok.eos_id]
            ids += head + body; mask += [0] * len(head) + [1] * len(body)
        else:
            part = [tok.user_id] + tok.encode('\n' + text + '\n'); ids += part; mask += [0] * len(part)
    if add_generation_prompt:
        head = [tok.assistant_id] + tok.encode('\n'); ids += head; mask += [0] * len(head)
    return ids, mask

def clean(v, max_chars=60000):
    if not isinstance(v, str):
        return ''
    v = v.replace('\x00', '').strip()
    if not v or _SECRET.search(v):
        return ''
    v = v[:max_chars]
    if '@' in v: v = _EMAIL.sub('[EMAIL]', v)
    if '998' in v: v = _PHONE.sub('[PHONE]', v)
    return _SPECIAL_RE.sub('', v) if '<|' in v else v

def stable_hash(x):
    return int(hashlib.md5(str(x).encode('utf-8', 'replace')).hexdigest()[:12], 16)

def hf_rows(repo, name=None, split='train', columns=None, limit=None):
    from datasets import load_dataset
    kw = dict(split=split, streaming=True, **({'name': name} if name else {}))
    try:
        ds = load_dataset(repo, columns=columns, **kw) if columns else load_dataset(repo, **kw)
    except (TypeError, ValueError):
        ds = load_dataset(repo, **kw)
    return islice(iter(ds), limit) if limit else iter(ds)

def normalize_turns(turns):
    out = []
    for role, text in turns:
        if out and out[-1][0] == role: out[-1][1] += '\n\n' + text
        else: out.append([role, text])
    while out and out[0][0] != 'user': out.pop(0)
    while out and out[-1][0] != 'assistant': out.pop()
    return out

UZ_CODE_TEMPLATES = ["Quyidagi dasturlash vazifasi uchun ishlaydigan Python funksiyasini yozing.\n\n{t}",
                     "Python'da yeching:\n\n{t}", "Vazifa:\n{t}\n\nFaqat Python kodini qaytaring.", "{t}\n\nYechimni Python'da yozing."]

# ---------------- manbalar ----------------
def gen_fw2():
    for r in hf_rows('HuggingFaceFW/fineweb-2', 'uzn_Latn', columns=['id', 'text'], limit=SMOKE_ROWS):
        t = clean(r.get('text'))
        if len(t) >= 200: yield {'pt': t, 'key': r.get('id')}

def gen_wiki():
    for r in hf_rows('wikimedia/wikipedia', '20231101.uz', limit=SMOKE_ROWS):
        t = clean(r.get('text'))
        if len(t) >= 500: yield {'pt': f"{r.get('title', '').strip()}\n\n{t}", 'key': r.get('id')}

def gen_uz_lm():
    for hf_split, split in (('train', 'train'), ('validation', 'val')):
        for r in hf_rows('ulugbek2013/uzbek-lang-lm', split=hf_split, limit=SMOKE_ROWS):
            turns = normalize_turns([[m['role'], clean(m.get('content'), 16000)] for m in (r.get('messages') or [])
                                     if m.get('role') in ('user', 'assistant') and clean(m.get('content'), 16000)])
            if not turns and r.get('instruction') and r.get('output'):
                turns = [['user', clean(r['instruction'], 16000)], ['assistant', clean(r['output'], 16000)]]
            if turns:
                yield {'chat': turns, 'split': split}
            elif len(clean(r.get('text'))) >= 50:
                yield {'pt': clean(r.get('text')), 'split': split}

def gen_code():
    import ast
    seen = set()
    for r in hf_rows('JetBrains-Research/OpenCodeInstruct-Clean', columns=['id', 'instruction', 'code', 'testcase', 'problem_group_id'],
                     limit=SMOKE_ROWS):
        g = r.get('problem_group_id') or r.get('id')
        code, ins = clean(r.get('code'), 20000), clean(r.get('instruction'), 8000)
        if not code or not ins or g in seen:   # har masala guruhidan bittadan (takrorlar ko'p)
            continue
        try: ast.parse(code)
        except (SyntaxError, ValueError): continue
        seen.add(g)
        split = 'val' if stable_hash(g) % 1000 < 5 else 'train'
        pt = '"""\n' + ins.replace('"""', "'''") + '\n"""\n\n' + code + '\n'
        prompt = UZ_CODE_TEMPLATES[stable_hash(g) % len(UZ_CODE_TEMPLATES)].format(t=ins)
        yield {'pt': pt, 'chat': [['user', prompt], ['assistant', code]], 'chat_pt': False, 'split': split}

def collect(raw):
    raw.mkdir(parents=True, exist_ok=True)
    stats, seen = Counter(), set()
    plan = [('fw2', USE_FW2, gen_fw2, FW2_MAX_CHARS), ('wiki', USE_WIKI, gen_wiki, WIKI_MAX_CHARS),
            ('uz_lm', USE_UZ_LM, gen_uz_lm, None), ('code', USE_CODE, gen_code, CODE_MAX_CHARS)]
    for name, on, gen, budget in plan:
        if not on: continue
        t0, n, chars, val_chars = time.time(), 0, 0, 0
        try:
            with (raw / f'{name}.jsonl').open('w', encoding='utf-8') as f:
                for rec in gen():
                    body = (rec.get('pt') or '') + json.dumps(rec.get('chat'), ensure_ascii=False)
                    h = hashlib.blake2b(body.encode('utf-8', 'replace'), digest_size=8).digest()
                    if h in seen: continue
                    seen.add(h)
                    if 'split' not in rec:
                        rec['split'] = 'val' if (stable_hash(rec.get('key', h)) % 10000 < VAL_PCT * 100 and val_chars < 3e6) else 'train'
                    size = len(rec.get('pt') or '') + sum(len(t) for _, t in rec.get('chat') or [])
                    val_chars += size if rec['split'] == 'val' else 0
                    rec.pop('key', None); f.write(json.dumps(rec, ensure_ascii=False) + '\n')
                    n += 1; chars += size
                    if n % 100_000 == 0: print(f'   {name}: {n:,} hujjat, {chars / 1e6:,.0f}M belgi', flush=True)
                    if budget and chars >= budget: break
        except Exception as e:  # noqa: BLE001
            print(f'⚠️ {name} uzildi: {repr(e)[:200]} (yig\'ilgani saqlandi)')
        stats[name] = chars
        print(f'✔ {name}: {n:,} hujjat, {chars / 1e6:,.1f}M belgi, {time.time() - t0:.0f}s', flush=True)
    return stats

def iter_raw(raw):
    for p in sorted(raw.glob('*.jsonl')):
        with p.open(encoding='utf-8') as f:
            for line in f:
                yield p.stem, json.loads(line)

def build_dataset():
    raw = Path(LOCAL_TMP); shutil.rmtree(raw, ignore_errors=True)
    t0 = time.time()
    stats = collect(raw)
    if sum(stats.values()) < (2e5 if PROFILE == 'smoke' else 5e7):
        raise RuntimeError("Data juda kam — Colab'da internet ishlayotganini tekshiring.")
    # tokenizer namunasi: har manbadan teng ulush (o'zbekcha so'zlar yaxshi bo'laklansin)
    def balanced_sample():
        per = TOKENIZER_SAMPLE_CHARS / max(1, len(stats))
        for p in sorted(raw.glob('*.jsonl')):
            got = 0
            with p.open(encoding='utf-8') as f:
                for line in f:
                    r = json.loads(line)
                    for t in [r.get('pt') or ''] + [x for _, x in r.get('chat') or []]:
                        if t: yield t; got += len(t)
                    if got > per: break
    tok = UzTokenizer.train(balanced_sample(), VOCAB)
    tok.tk.save(str(DATA / 'tokenizer.json'))
    for s in ("Salom, O'zbekiston! G'alaba, ma'no 🧠", "def f(x):\n    return x * 2\n"):
        assert tok.decode(tok.encode(s)) == s
    print(f'Tokenizer: vocab={tok.vocab_size}, {time.time() - t0:.0f}s | namuna:',
          [tok.tk.id_to_token(i) for i in tok.encode("O'zbekiston g'alabasi")])
    files = {s: (DATA / f'{s}.bin').open('wb') for s in ('train', 'val')}
    sft = {s: dict(t=(DATA / f'sft_{s}_tokens.bin').open('wb'), m=(DATA / f'sft_{s}_mask.bin').open('wb'), idx=[], pos=0)
           for s in ('train', 'val')}
    ntok, nchar, nsft, batch = Counter(), Counter(), Counter(), []
    def flush(batch):
        pts = [r for _, r in batch if r.get('pt')]
        for r, ids in zip(pts, tok.encode_batch([r['pt'] for r in pts])):
            np.asarray([tok.bos_id] + ids + [tok.eos_id], np.uint16).tofile(files[r['split']])
            ntok[r['split']] += len(ids) + 2; nchar[r['split']] += len(r['pt'])
        for src, r in batch:
            if not r.get('chat'): continue
            ids, mask = chat_ids(tok, r['chat'])
            if r.get('chat_pt', True):     # chat matni pretrainda ham (til modeli sifatida)
                np.asarray(ids, np.uint16).tofile(files[r['split']]); ntok[r['split']] += len(ids)
            if len(ids) <= SEQ + 2:
                S = sft[r['split']]
                np.asarray(ids, np.uint16).tofile(S['t']); np.asarray(mask, np.uint8).tofile(S['m'])
                S['idx'].append((S['pos'], len(ids))); S['pos'] += len(ids); nsft[(src, r['split'])] += 1
    for item in iter_raw(raw):
        batch.append(item)
        if len(batch) >= 2048: flush(batch); batch = []
    flush(batch)
    for f in files.values(): f.close()
    for s, S in sft.items():
        S['t'].close(); S['m'].close(); np.save(DATA / f'sft_{s}_index.npy', np.asarray(S['idx'], np.int64).reshape(-1, 2))
    meta = {'vocab': tok.vocab_size, 'tokens': dict(ntok), 'chars_per_token': round(sum(nchar.values()) / max(1, sum(ntok.values())), 3),
            'sft': {f'{a}|{b}': v for (a, b), v in nsft.items()}, 'source_chars': dict(stats), 'seq': SEQ}
    meta['prep_id'] = hashlib.sha256((DATA / 'tokenizer.json').read_bytes() + json.dumps(meta, sort_keys=True).encode()).hexdigest()[:12]
    (DATA / 'meta.json').write_text(json.dumps(meta, indent=2, ensure_ascii=False))
    shutil.rmtree(raw, ignore_errors=True)
    print(f'Data tayyor: {(time.time() - t0) / 60:.1f} daqiqa')
    return tok, meta

if (DATA / 'meta.json').exists():
    tok, META = UzTokenizer.load(DATA / 'tokenizer.json'), json.loads((DATA / 'meta.json').read_text())
    print('✅ Tayyor data topildi (qayta yuklanmaydi):', DATA)
else:
    tok, META = build_dataset()
TRAIN_TOK = np.memmap(DATA / 'train.bin', dtype=np.uint16, mode='r')
VAL_TOK = np.memmap(DATA / 'val.bin', dtype=np.uint16, mode='r')
SFT = {s: dict(t=np.memmap(DATA / f'sft_{s}_tokens.bin', dtype=np.uint16, mode='r'),
               m=np.memmap(DATA / f'sft_{s}_mask.bin', dtype=np.uint8, mode='r'),
               idx=np.load(DATA / f'sft_{s}_index.npy')) for s in ('train', 'val')}
print(f"Pretrain: {len(TRAIN_TOK) / 1e6:,.1f}M train / {len(VAL_TOK) / 1e6:,.2f}M val token | belgi/token: {META['chars_per_token']}")
print(f"SFT: {len(SFT['train']['idx']):,} train / {len(SFT['val']['idx']):,} val chat | {META['sft']}")

## 3. Model
nanoGPT kabi bitta faylda, sof JAX (Flax'siz) — har bir qatorini o'qib tushunsa bo'ladi.
- **Blok** (Llama): pre-RMSNorm → attention → pre-RMSNorm → FFN, residual.
- **MLA** (DeepSeek): K/V `MLA_KV_RANK` o'lchamli latentga siqiladi (inference KV-cache ~8× kichik), RoPE alohida 32 o'lchamda, QK-norm bilan.
- **DeepSeekMoE**: birinchi qatlam dense, qolganlarida 1 shared + 16 ta mayda ekspert, har token uchun top-2. Router sigmoid; ekspert tanlashga **bias** qo'shiladi va har qadamda yuklamaga qarab yangilanadi (aux-loss-free). Statik sig'im (`CAPACITY_FACTOR`) — TPU uchun shakllar o'zgarmaydi.
- **Doc-masking**: har `<|bos|>` dan yangi hujjat — token boshqa hujjatga qaramaydi.
- **Chunked CE** + logit soft-capping; `REMAT` bilan xotira tejaladi.

In [ ]:
# 2. Model: nanoGPT tuzilmasi + Llama bloklari + DeepSeek MLA/MoE/MTP (sof JAX)
DT = jnp.bfloat16 if PLATFORM == 'tpu' else jnp.float32   # TPU: bf16 hisob, fp32 master vaznlar
F32 = jnp.float32
BOS, PAD = tok.bos_id, tok.pad_id
V = tok.vocab_size
N_MOE = N_LAYER - N_DENSE_LAYERS
HD = D_MODEL // N_HEAD
Q_DIM = (MLA_QK_NOPE + MLA_QK_ROPE) if ATTN == 'mla' else HD     # q/k bosh o'lchami
V_DIM = MLA_V_DIM if ATTN == 'mla' else HD

def rms(x, w, eps=1e-6):
    xf = x.astype(F32)
    return (xf * jax.lax.rsqrt(jnp.mean(xf * xf, -1, keepdims=True) + eps) * w.astype(F32)).astype(x.dtype)

def rope_tables(T, dim):
    inv = 1.0 / (ROPE_THETA ** (np.arange(0, dim, 2, dtype=np.float64) / dim))
    f = np.outer(np.arange(T), inv)
    return jnp.asarray(np.cos(f), F32), jnp.asarray(np.sin(f), F32)

ROPE_DIM = MLA_QK_ROPE if ATTN == 'mla' else HD
COS, SIN = rope_tables(SEQ, ROPE_DIM)

def rope(x, cos, sin):          # x [B,T,H,D], Llama/HF "rotate_half" konvensiyasi
    d = x.shape[-1] // 2
    x1, x2 = x[..., :d].astype(F32), x[..., d:].astype(F32)
    c, s = cos[None, :, None, :], sin[None, :, None, :]
    return jnp.concatenate([x1 * c - x2 * s, x2 * c + x1 * s], -1).astype(x.dtype)

def lin(x, w):
    return jnp.einsum('...i,io->...o', x, w.astype(DT))

# ---------------- parametrlar ----------------
def init_params(key):
    ks = iter(jax.random.split(key, 4096))
    def W(*shape, zero=False):        # modded-nanoGPT: std = 0.5/sqrt(fan_in); chiqish proyeksiyalari 0
        if zero: return jnp.zeros(shape, F32)
        return jax.random.normal(next(ks), shape, F32) * (0.5 / math.sqrt(shape[-2]))
    ones = lambda n: jnp.ones((n,), F32)
    def attn_p():
        if ATTN == 'mla':
            return {'wq': W(D_MODEL, N_HEAD * Q_DIM), 'wkv_a': W(D_MODEL, MLA_KV_RANK + MLA_QK_ROPE),
                    'kv_norm': ones(MLA_KV_RANK), 'wkv_b': W(MLA_KV_RANK, N_HEAD * (MLA_QK_NOPE + MLA_V_DIM)),
                    'q_norm': ones(Q_DIM), 'k_norm': ones(Q_DIM), 'wo': W(N_HEAD * V_DIM, D_MODEL, zero=True)}
        return {'wq': W(D_MODEL, N_HEAD * HD), 'wk': W(D_MODEL, GQA_KV_HEADS * HD), 'wv': W(D_MODEL, GQA_KV_HEADS * HD),
                'q_norm': ones(HD), 'k_norm': ones(HD), 'wo': W(N_HEAD * HD, D_MODEL, zero=True)}
    def mlp_p(h):
        return {'w_gate': W(D_MODEL, h), 'w_up': W(D_MODEL, h), 'w_down': W(h, D_MODEL, zero=True)}
    def moe_p():
        E, h = N_EXPERTS, EXPERT_HIDDEN
        return {'router': jax.random.normal(next(ks), (D_MODEL, E), F32) * 0.02,
                'e_gate': W(E, D_MODEL, h), 'e_up': W(E, D_MODEL, h), 'e_down': W(E, h, D_MODEL, zero=True),
                'shared': mlp_p(h * N_SHARED)}
    layers = [{'norm1': ones(D_MODEL), 'norm2': ones(D_MODEL), 'attn': attn_p(),
               'ffn': mlp_p(DENSE_HIDDEN) if i < N_DENSE_LAYERS else moe_p()} for i in range(N_LAYER)]
    p = {'embed': jax.random.normal(next(ks), (V, D_MODEL), F32) * 0.02, 'layers': layers,
         'norm_f': ones(D_MODEL), 'head': W(D_MODEL, V, zero=True)}
    if MTP_WEIGHT > 0:
        p['mtp'] = {'norm_h': ones(D_MODEL), 'norm_e': ones(D_MODEL), 'proj': W(2 * D_MODEL, D_MODEL),
                    'block': {'norm1': ones(D_MODEL), 'norm2': ones(D_MODEL), 'attn': attn_p(), 'ffn': mlp_p(DENSE_HIDDEN)},
                    'norm': ones(D_MODEL)}
    return p

# ---------------- qismlar ----------------
def attention(p, x, seg):
    B, T, _ = x.shape
    cos, sin = COS[:T], SIN[:T]
    if ATTN == 'mla':     # DeepSeek MLA: KV past o'lchamli latentga siqiladi, RoPE alohida (decoupled) qism
        q = lin(x, p['wq']).reshape(B, T, N_HEAD, Q_DIM)
        q = jnp.concatenate([q[..., :MLA_QK_NOPE], rope(q[..., MLA_QK_NOPE:], cos, sin)], -1)
        kv_a = lin(x, p['wkv_a'])
        c_kv, k_rope = rms(kv_a[..., :MLA_KV_RANK], p['kv_norm']), kv_a[..., MLA_KV_RANK:]
        kv = lin(c_kv, p['wkv_b']).reshape(B, T, N_HEAD, MLA_QK_NOPE + MLA_V_DIM)
        k_rope = jnp.broadcast_to(rope(k_rope[:, :, None, :], cos, sin), (B, T, N_HEAD, MLA_QK_ROPE))
        k = jnp.concatenate([kv[..., :MLA_QK_NOPE], k_rope], -1)
        v = kv[..., MLA_QK_NOPE:]
    else:                 # Llama GQA
        q = rope(lin(x, p['wq']).reshape(B, T, N_HEAD, HD), cos, sin)
        k = rope(lin(x, p['wk']).reshape(B, T, GQA_KV_HEADS, HD), cos, sin)
        v = lin(x, p['wv']).reshape(B, T, GQA_KV_HEADS, HD)
        rep = N_HEAD // GQA_KV_HEADS
        k, v = jnp.repeat(k, rep, axis=2), jnp.repeat(v, rep, axis=2)
    q, k = rms(q, p['q_norm']), rms(k, p['k_norm'])        # QK-norm: Muon bilan barqaror trening
    logits = jnp.einsum('bthd,bshd->bhts', q, k, preferred_element_type=F32) * (Q_DIM ** -0.5)
    causal = jnp.tril(jnp.ones((T, T), bool))
    mask = causal[None, None] & (seg[:, None, :, None] == seg[:, None, None, :])   # hujjat ichida attention
    probs = jax.nn.softmax(jnp.where(mask, logits, -1e30), axis=-1).astype(DT)
    out = jnp.einsum('bhts,bshd->bthd', probs, v).reshape(B, T, N_HEAD * V_DIM)
    return lin(out, p['wo'])

def swiglu(p, x):
    return lin(jax.nn.silu(lin(x, p['w_gate'])) * lin(x, p['w_up']), p['w_down'])

def capacity(n_tokens, cf):
    if cf is None:                       # dropless (inference): har ekspert hamma tokenni sig'dira oladi
        return n_tokens
    c = int(math.ceil(n_tokens * TOP_K * cf / N_EXPERTS))
    return min(n_tokens, max(8, (c + 7) // 8 * 8))

def moe(p, x, bias, cf):
    """DeepSeekMoE: sigmoid affinity, bias bilan top-k tanlash (aux-loss-free), 1 shared + fine-grained ekspertlar.
    Dispatch: sort/scatter'siz — cumsum orqali joy, gather bilan yig'ish (TPU uchun statik shakllar)."""
    B, T, d = x.shape
    N, E, K = B * T, N_EXPERTS, TOP_K
    xf = x.reshape(N, d)
    s = jax.nn.sigmoid(jnp.einsum('nd,de->ne', xf.astype(F32), p['router']))       # [N,E]
    _, idx = jax.lax.top_k(s + bias[None, :], K)                                      # bias faqat TANLASHga ta'sir qiladi
    w = jnp.take_along_axis(s, idx, 1)
    w = w / (w.sum(-1, keepdims=True) + 1e-9)
    C = capacity(N, cf)
    e = idx.reshape(-1)                                                              # [A=N*K]
    oh = jax.nn.one_hot(e, E, dtype=jnp.int32)
    pos = jnp.sum(jnp.cumsum(oh, 0) * oh, -1) - 1
    keep = pos < C
    slot = jnp.where(keep, e * C + pos, E * C)
    tok_of = jnp.full((E * C + 1,), N, jnp.int32).at[slot].set(jnp.repeat(jnp.arange(N, dtype=jnp.int32), K))[:E * C]
    xin = jnp.concatenate([xf, jnp.zeros((1, d), xf.dtype)], 0)[tok_of].reshape(E, C, d)
    g = jnp.einsum('ecd,edh->ech', xin, p['e_gate'].astype(DT))
    u = jnp.einsum('ecd,edh->ech', xin, p['e_up'].astype(DT))
    out = jnp.einsum('ech,ehd->ecd', jax.nn.silu(g) * u, p['e_down'].astype(DT)).reshape(E * C, d)
    out = jnp.concatenate([out, jnp.zeros((1, d), out.dtype)], 0)[slot]
    y = (out * (w.reshape(-1) * keep).astype(out.dtype)[:, None]).reshape(N, K, d).sum(1)
    y = y + swiglu(p['shared'], xf)
    load = oh.sum(0).astype(F32) / (N * K)                     # har ekspert ulushi (ideal: 1/E)
    P = (s / s.sum(-1, keepdims=True)).mean(0)
    bal = jnp.sum(load * E * P)                                # DeepSeek-V3 uslubidagi balans loss (batch bo'yicha)
    drop = 1.0 - keep.mean()
    return y.reshape(B, T, d), load, bal, drop

def block(lp, x, seg, bias, cf, is_moe):
    x = x + attention(lp['attn'], rms(x, lp['norm1']), seg)
    h = rms(x, lp['norm2'])
    if is_moe:
        y, load, bal, drop = moe(lp['ffn'], h, bias, cf)
    else:
        y, load, bal, drop = swiglu(lp['ffn'], h), jnp.zeros((N_EXPERTS,), F32), jnp.zeros((), F32), jnp.zeros((), F32)
    return x + y, load, bal, drop

def forward(params, bias, tokens, cf):
    """tokens [B,T] -> yakuniy normdan oldingi yashirin holat, MoE statistikasi."""
    x = params['embed'][tokens].astype(DT)
    seg = jnp.cumsum(tokens == BOS, axis=1)
    loads, bals, drops = [], [], []
    for i, lp in enumerate(params['layers']):
        is_moe = i >= N_DENSE_LAYERS
        f = functools.partial(block, cf=cf, is_moe=is_moe)
        if REMAT:
            f = jax.checkpoint(f, policy=jax.checkpoint_policies.nothing_saveable)
        x, load, bal, drop = f(lp, x, seg, bias[i - N_DENSE_LAYERS] if is_moe else bias[0])
        if is_moe:
            loads.append(load); bals.append(bal); drops.append(drop)
    stats = (jnp.stack(loads) if loads else jnp.zeros((1, N_EXPERTS), F32),
             sum(bals) / max(1, len(bals)), sum(drops) / max(1, len(drops)))
    return x, seg, stats

def head_logits(params, h):
    logits = lin(h, params['head']).astype(F32)
    return SOFTCAP * jnp.tanh(logits / SOFTCAP) if SOFTCAP else logits    # Gemma 2: logit soft-capping

def chunked_ce(params, h, tgt, mask):
    """[B,T,d] -> o'rtacha CE. Har ketma-ketlik alohida (to'liq [B*T, V] logits xotirada saqlanmaydi)."""
    @jax.checkpoint
    def body(acc, xs):
        hh, tt, mm = xs
        lg = head_logits(params, hh)
        nll = jax.nn.logsumexp(lg, -1) - jnp.take_along_axis(lg, tt[:, None], -1)[:, 0]
        return acc + jnp.sum(nll * mm), None
    tot, _ = jax.lax.scan(body, jnp.zeros((), F32), (h, tgt, mask.astype(F32)))
    return tot / jnp.maximum(mask.sum(), 1.0)

def mtp_hidden(params, x, seg, next_tokens):
    """DeepSeek-V3 MTP: h_t va Emb(x_{t+1}) -> bitta blok -> x_{t+2} bashorati."""
    p = params['mtp']
    e = params['embed'][next_tokens].astype(DT)
    z = lin(jnp.concatenate([rms(x, p['norm_h']), rms(e, p['norm_e'])], -1), p['proj'])
    z, _, _, _ = block(p['block'], z, seg, jnp.zeros((N_EXPERTS,), F32), None, False)
    return rms(z, p['norm'])

def loss_fn(params, bias, tokens, mask, cf):
    """tokens [B, SEQ+2], mask [B, SEQ+1] (qaysi keyingi-token o'qitiladi)."""
    inp, tgt, tgt2 = tokens[:, :SEQ], tokens[:, 1:SEQ + 1], tokens[:, 2:SEQ + 2]
    x, seg, (loads, bal, drop) = forward(params, bias, inp, cf)
    ce = chunked_ce(params, rms(x, params['norm_f']), tgt, mask[:, :SEQ])
    loss = ce + (BALANCE_LOSS * bal if N_MOE else 0.0)
    if MTP_WEIGHT > 0:
        z = mtp_hidden(params, x, seg, tgt)
        loss = loss + MTP_WEIGHT * chunked_ce(params, z, tgt2, mask[:, 1:SEQ + 1])
    return loss, {'ce': ce, 'load': loads, 'drop': drop}

def param_counts(params):
    tot = sum(int(np.prod(x.shape)) for x in jax.tree_util.tree_leaves(params))
    emb = int(np.prod(params['embed'].shape))
    expert = sum(int(np.prod(l['ffn'][k].shape)) for l in params['layers'][N_DENSE_LAYERS:] for k in ('e_gate', 'e_up', 'e_down'))
    mtp = sum(int(np.prod(x.shape)) for x in jax.tree_util.tree_leaves(params.get('mtp', {})))
    active = tot - emb - mtp - expert + expert * TOP_K // max(1, N_EXPERTS)
    return tot, active

KEY = jax.random.PRNGKey(SEED)
_shapes = jax.eval_shape(init_params, KEY)
TOTAL_P, ACTIVE_P = param_counts(_shapes)
FLOPS_PER_TOKEN = 6 * ACTIVE_P + 6 * N_LAYER * SEQ * N_HEAD * (Q_DIM + V_DIM) // 2   # model FLOPs (remat'siz)
print(f'Parametrlar: jami {TOTAL_P / 1e6:.1f}M | har tokenda faol {ACTIVE_P / 1e6:.1f}M (embedding\'siz) | '
      f'{N_LAYER} qatlam ({N_DENSE_LAYERS} dense + {N_MOE} MoE), {ATTN.upper()}, d={D_MODEL}')

## 4. Optimizer: Muon + AdamW
**Muon** (Keller Jordan; modded-nanoGPT speedrun'ida nanoGPT'ni ko'p marta tezlashtirgan, Moonshot "Moonlight" MoE modelida katta miqyosda sinalgan): momentum gradientini Newton–Schulz iteratsiyasi bilan ortogonallashtiradi. Barcha yashirin matritsalarga (attention, FFN, **har bir ekspert alohida**) qo'llanadi. Embedding, head, normlar va router — AdamW. Barcha qadam `jit` + `lax.scan` (gradient akkumulyatsiyasi) ichida bitta XLA dasturi bo'lib kompilyatsiya qilinadi; HBM ga sig'masa micro-batch avtomatik kichrayadi.

In [ ]:
# 3. Optimizer (Muon + AdamW), jit train/eval qadamlari, data batchlari, checkpoint
from typing import NamedTuple
from jax.sharding import Mesh, NamedSharding, PartitionSpec as P

class MuonState(NamedTuple):
    mu: optax.Params

def newton_schulz(G, steps=5):
    """Muon: gradientni (taxminan) ortogonal matritsaga aylantiradi — barcha yo'nalishlar teng o'rganiladi."""
    a, b, c = 3.4445, -4.7750, 2.0315
    X = G.astype(jnp.bfloat16)
    tr = X.shape[-2] > X.shape[-1]
    if tr:
        X = jnp.swapaxes(X, -1, -2)
    X = X / (jnp.sqrt(jnp.sum(jnp.square(X.astype(F32)), axis=(-2, -1), keepdims=True)) + 1e-7).astype(X.dtype)
    for _ in range(steps):
        A = X @ jnp.swapaxes(X, -1, -2)
        X = a * X + (b * A + c * (A @ A)) @ X
    if tr:
        X = jnp.swapaxes(X, -1, -2)
    return X.astype(G.dtype)

def scale_by_muon(momentum=0.95, nesterov=True):
    def init(params):
        return MuonState(jax.tree_util.tree_map(jnp.zeros_like, params))
    def update(g, state, params=None):
        mu = jax.tree_util.tree_map(lambda m, x: momentum * m + x, state.mu, g)
        d = jax.tree_util.tree_map(lambda m, x: x + momentum * m, mu, g) if nesterov else mu
        # [.., kirish, chiqish] shakl: masshtab sqrt(max(1, chiqish/kirish)) (Keller Jordan, modded-nanoGPT)
        d = jax.tree_util.tree_map(lambda x: newton_schulz(x) * math.sqrt(max(1.0, x.shape[-1] / x.shape[-2])), d)
        return d, MuonState(mu)
    return optax.GradientTransformation(init, update)

def label_params(params):
    def lab(path, x):
        name = str(getattr(path[-1], 'key', ''))
        return 'adam' if (x.ndim < 2 or name in ('embed', 'head', 'router')) else 'muon'
    return jax.tree_util.tree_map_with_path(lab, params)

_partition = getattr(optax, 'partition', None) or optax.multi_transform
def make_tx(scale=1.0):
    return _partition({
        'muon': optax.chain(scale_by_muon(MUON_MOMENTUM), optax.add_decayed_weights(WEIGHT_DECAY), optax.scale(-MUON_LR * scale)),
        'adam': optax.chain(optax.scale_by_adam(*ADAM_BETAS, eps=1e-10), optax.add_decayed_weights(WEIGHT_DECAY),
                            optax.scale(-ADAM_LR * scale)),
    }, label_params)

# --- Qurilmalar bo'yicha data-parallel (TPU v5e-1 da 1 ta qurilma; v5e-8 / v2-8 da avtomatik) ---
MESH = Mesh(np.array(DEVICES), ('data',))
REPL = NamedSharding(MESH, P())
BATCH_SH = NamedSharding(MESH, P(None, 'data'))     # [ACCUM, B, ...] -> B o'qi bo'yicha bo'linadi

def make_train_step(tx):
    def train_step(params, opt_state, bias, tokens, mask, lr_factor):
        def micro(carry, xs):
            g_acc, ce_acc, load_acc, drop_acc = carry
            (_, aux), g = jax.value_and_grad(loss_fn, has_aux=True)(params, bias, xs[0], xs[1], CAPACITY_FACTOR)
            g_acc = jax.tree_util.tree_map(lambda a, b: a + b.astype(F32), g_acc, g)
            return (g_acc, ce_acc + aux['ce'], load_acc + aux['load'], drop_acc + aux['drop']), None
        n = tokens.shape[0]
        zero = (jax.tree_util.tree_map(jnp.zeros_like, params), jnp.zeros((), F32), jnp.zeros_like(bias), jnp.zeros((), F32))
        (g, ce, load, drop), _ = jax.lax.scan(micro, zero, (tokens, mask))
        g = jax.tree_util.tree_map(lambda x: x / n, g)
        gnorm = optax.global_norm(g)
        g = jax.tree_util.tree_map(lambda x: x * jnp.minimum(1.0, GRAD_CLIP / (gnorm + 1e-6)), g)
        upd, opt_state = tx.update(g, opt_state, params)
        params = optax.apply_updates(params, jax.tree_util.tree_map(lambda u: u * lr_factor, upd))
        load = load / n
        # DeepSeek-V3 aux-loss-free balans: kam yuklangan ekspert biasi oshadi, ko'p yuklangan — kamayadi
        bias = bias + BIAS_UPDATE * jnp.sign(load.mean(-1, keepdims=True) - load)
        return params, opt_state, bias, {'ce': ce / n, 'gnorm': gnorm, 'load_max': load.max() * N_EXPERTS,
                                         'load_min': load.min() * N_EXPERTS, 'drop': drop / n}
    return jax.jit(train_step, donate_argnums=(0, 1, 2))

@jax.jit
def eval_step(params, bias, tokens, mask):
    _, aux = loss_fn(params, bias, tokens, mask, 2.0)
    return aux['ce']

def compile_fit(step_fn, params, opt_state, bias, micro, accum, label):
    """AOT kompilyatsiya; HBM ga sig'masa micro-batch yarmiga tushadi (global batch ACCUM orqali saqlanadi)."""
    nd = len(DEVICES)
    while True:
        micro = max(nd, micro // nd * nd)
        tok_s = jax.ShapeDtypeStruct((accum, micro, SEQ + 2), jnp.int32, sharding=BATCH_SH)
        msk_s = jax.ShapeDtypeStruct((accum, micro, SEQ + 1), jnp.float32, sharding=BATCH_SH)
        t0 = time.time()
        try:
            c = step_fn.lower(params, opt_state, bias, tok_s, msk_s, jnp.float32(1.0)).compile()
            print(f'[{label}] kompilyatsiya: micro={micro} accum={accum} ({time.time() - t0:.0f}s)')
            return c, micro, accum
        except Exception as e:  # noqa: BLE001
            msg = str(e)
            if micro > nd and any(k in msg for k in ('RESOURCE_EXHAUSTED', 'out of memory', 'Out of memory', 'exceeds')):
                print(f'[{label}] micro={micro} sig\'madi -> {micro // 2}')
                micro //= 2; accum *= 2
                continue
            raise

def pretrain_batch(rng, src, micro, accum):
    L = SEQ + 2
    starts = rng.integers(0, len(src) - L, size=(accum * micro,))
    toks = np.stack([src[s:s + L] for s in starts]).astype(np.int32).reshape(accum, micro, L)
    return toks, np.ones((accum, micro, SEQ + 1), np.float32)

def sft_batch(rng, d, micro, accum, sequential_from=None):
    """Chat misollarini qatorlarga joylash (packing); loss faqat javob tokenlarida."""
    L = SEQ + 2
    toks = np.full((accum * micro, L), PAD, np.int32)
    msk = np.zeros((accum * micro, L), np.float32)
    k = sequential_from
    for i in range(accum * micro):
        pos = 0
        while pos < L:
            if k is None:
                j = int(rng.integers(0, len(d['idx'])))
            else:
                j = k % len(d['idx']); k += 1
            s, n = (int(v) for v in d['idx'][j])
            if n > L - pos:
                if pos: break
                n = L
            toks[i, pos:pos + n] = d['t'][s:s + n]; msk[i, pos:pos + n] = d['m'][s:s + n]; pos += n
    return toks.reshape(accum, micro, L), msk[:, 1:].reshape(accum, micro, SEQ + 1)

def put(x):
    return jax.device_put(x, BATCH_SH)

def evaluate(params, bias, kind):
    rng, losses = np.random.default_rng(1234), []
    for i in range(EVAL_BATCHES):
        if kind == 'pretrain':
            t, m = pretrain_batch(rng, VAL_TOK, MICRO_BATCH, 1)
        else:
            t, m = sft_batch(rng, SFT['val'], SFT_MICRO_BATCH, 1, sequential_from=i * SFT_MICRO_BATCH * 4)
        losses.append(float(eval_step(params, bias, jax.device_put(t[0], REPL), jax.device_put(m[0], REPL))))
    return float(np.mean(losses))

MODEL_KEYS = ['D_MODEL', 'N_LAYER', 'N_HEAD', 'ATTN', 'MLA_KV_RANK', 'MLA_QK_NOPE', 'MLA_QK_ROPE', 'MLA_V_DIM', 'GQA_KV_HEADS',
              'N_DENSE_LAYERS', 'DENSE_HIDDEN', 'N_EXPERTS', 'TOP_K', 'N_SHARED', 'EXPERT_HIDDEN', 'MTP_WEIGHT', 'SEQ', 'VOCAB']
MODEL_SPEC = {k: globals()[k] for k in MODEL_KEYS}

# ---------------- .nit fayl formati (nanoGPT-uz) ----------------
# [b'NIT1'][uint64: header uzunligi][JSON header: meta + tensorlar indeksi][64 baytga tekislangan xom tensorlar]
# pickle ishlatilmaydi: .nit faylni ochish kod bajarmaydi (safetensors kabi xavfsiz), Drive'da memmap bilan o'qiladi.
NIT_MAGIC = b'NIT1'

def nit_save(path, tensors, meta):
    path = Path(path); tmp = path.with_name(path.name + '.tmp')
    index, off, arrs = {}, 0, []
    for name, a in tensors.items():
        a = np.ascontiguousarray(np.asarray(jax.device_get(a)))
        index[name] = {'dtype': str(a.dtype), 'shape': list(a.shape), 'offset': off, 'nbytes': int(a.nbytes)}
        arrs.append(a); off += (a.nbytes + 63) // 64 * 64
    header = json.dumps({'format': 'nanogpt-uz', 'version': 1, 'meta': meta, 'tensors': index}, ensure_ascii=False).encode()
    header += b' ' * ((64 - (12 + len(header)) % 64) % 64)
    with tmp.open('wb') as f:
        f.write(NIT_MAGIC); f.write(np.uint64(len(header)).tobytes()); f.write(header)
        for a in arrs:
            f.write(a.tobytes()); f.write(b'\0' * ((64 - a.nbytes % 64) % 64))
    os.replace(tmp, path)

def nit_load(path):
    raw = np.memmap(path, dtype=np.uint8, mode='r')
    if bytes(raw[:4]) != NIT_MAGIC:
        raise ValueError(f'{path}: .nit fayl emas')
    hl = int(np.frombuffer(raw[4:12], np.uint64)[0])
    head, base = json.loads(bytes(raw[12:12 + hl])), 12 + hl
    out = {n: np.frombuffer(raw[base + t['offset']: base + t['offset'] + t['nbytes']], dtype=jnp.dtype(t['dtype'])).reshape(t['shape'])
           for n, t in head['tensors'].items()}
    return head['meta'], out

def tree_to_named(prefix, tree):
    return {prefix + jax.tree_util.keystr(k): v for k, v in jax.tree_util.tree_flatten_with_path(tree)[0]}

def named_to_tree(prefix, named, ref):
    paths, treedef = jax.tree_util.tree_flatten_with_path(ref)
    return jax.tree_util.tree_unflatten(treedef, [named[prefix + jax.tree_util.keystr(k)] for k, _ in paths])

def save_ckpt(name, params, opt_state, bias, state):
    path, t0 = CKPT / f'{name}.nit', time.time()
    nit_save(path, tree_to_named('params', params) | tree_to_named('opt', opt_state) | {'bias': bias},
             {'kind': 'checkpoint', 'stage': name, 'state': state, 'spec': MODEL_SPEC, 'prep_id': META['prep_id']})
    print(f'💾 {path.name} ({path.stat().st_size / 2**30:.2f} GiB, {time.time() - t0:.0f}s)')

def load_ckpt(name, tx):
    path = CKPT / f'{name}.nit'
    if not path.exists():
        return None
    meta, t = nit_load(path)
    if meta.get('spec') != MODEL_SPEC or meta.get('prep_id') != META['prep_id']:
        raise RuntimeError(f'{path} boshqa sozlama/data bilan yaratilgan. Model sozlamalarini o\'zgartirmang '
                           f'yoki WORKDIR ni almashtiring.\n checkpoint: {meta.get("spec")}\n hozirgi: {MODEL_SPEC}')
    pref = jax.eval_shape(init_params, KEY)
    return {'params': named_to_tree('params', t, pref), 'opt_state': named_to_tree('opt', t, jax.eval_shape(tx.init, pref)),
            'bias': t['bias'], 'state': meta['state']}

def lr_factor(step, progress, warmup, decay_frac, min_ratio):
    w = min(1.0, step / max(1, warmup))
    start = 1.0 - decay_frac
    f = 1.0 if progress < start else max(min_ratio, 1.0 - (progress - start) / decay_frac * (1.0 - min_ratio))
    return w * f
print('Optimizer: Muon (yashirin matritsalar, MoE ekspertlari ham) + AdamW (embedding, head, norm, router) | qurilmalar:', len(DEVICES))

## 5. Pretrain
Logda: `loss`, `tok/s`, **MFU** (TPU cho'qqi quvvatidan foydalanish), ekspertlar yuklamasi (`min/max` 1.0 ga yaqin bo'lsa — balans yaxshi) va `drop` (sig'imdan oshgan tokenlar ulushi). LR jadvali WSD: warmup → doimiy → oxirgi 30% da 0 gacha.

In [ ]:
# 4. Pretrain (vaqt byudjeti bilan; Colab uzilsa checkpointdan davom etadi)
def run_stage(name, minutes, tx, init_params_fn, batch_fn, micro, accum, lr_scale_note, warmup):
    ck = load_ckpt(name, tx)
    if ck is not None:
        params, opt_state, bias = ck['params'], ck['opt_state'], ck['bias']
        st = ck['state']
        print(f"RESUME {name}: step={st['step']}, {st['elapsed'] / 60:.1f}/{minutes} daqiqa, {st['tokens'] / 1e9:.3f}B token")
        if st['elapsed'] >= minutes * 60:     # byudjet oshirilsa (MINUTES kattalashtirilsa) trening davom etadi
            return jax.device_put(params, REPL), jax.device_put(bias, REPL), st
    else:
        params, bias = init_params_fn()
        opt_state = tx.init(params)
        st = {'step': 0, 'elapsed': 0.0, 'tokens': 0, 'best_val': float('inf'), 'complete': False, 'history': []}
    params, opt_state, bias = (jax.device_put(x, REPL) for x in (params, opt_state, bias))
    step_fn, micro, accum = compile_fit(make_train_step(tx), params, opt_state, bias, micro, accum, name)
    tok_step = micro * accum * SEQ
    budget = minutes * 60
    rng = np.random.default_rng([SEED, st['step'], {'pretrain': 1, 'sft': 2}[name]])   # resume'da yangi namunalar
    t_start, el0 = time.time(), st['elapsed']
    last_eval = last_save = last_log = time.time()
    log_tok, metrics = 0, None
    elapsed = lambda: el0 + time.time() - t_start
    print(f'[{name}] {tok_step:,} token/qadam | byudjet {minutes} daqiqa | {lr_scale_note}')
    while True:
        prog = elapsed() / budget
        lrf = lr_factor(st['step'] + 1, prog, warmup, DECAY_FRAC, MIN_LR_RATIO)
        t, m = batch_fn(rng, micro, accum)
        params, opt_state, bias, metrics = step_fn(params, opt_state, bias, put(t), put(m), jnp.float32(lrf))
        st['step'] += 1; st['tokens'] += tok_step; log_tok += tok_step
        now = time.time()
        if st['step'] == 1 or st['step'] % LOG_EVERY == 0:
            mt = {k: float(v) for k, v in metrics.items()}       # shu yerda sinxronlanadi
            dt = time.time() - last_log; tps = log_tok / max(dt, 1e-6)
            mfu = f' MFU={100 * tps * FLOPS_PER_TOKEN / PEAK_FLOPS / len(DEVICES):.0f}%' if PEAK_FLOPS else ''
            moe_s = f" ekspert yuk[min/max]={mt['load_min']:.2f}/{mt['load_max']:.2f} drop={100 * mt['drop']:.1f}%" if N_MOE else ''
            print(f"{name} step={st['step']} loss={mt['ce']:.4f} lr×{lrf:.3f} gnorm={mt['gnorm']:.2f} tok/s={tps:,.0f}{mfu}"
                  f"{moe_s} | {elapsed() / 60:.1f}/{minutes} daq", flush=True)
            last_log, log_tok = time.time(), 0
        done = elapsed() >= budget
        if done or now - last_eval >= EVAL_EVERY_MIN * 60:
            v = evaluate(params, bias, 'pretrain' if name == 'pretrain' else 'sft')
            best = v < st['best_val']
            st['best_val'] = min(st['best_val'], v)
            st['history'].append((st['step'], round(elapsed() / 60, 2), v))
            print(f"VALIDATION {name}: step={st['step']} val_loss={v:.4f} ppl={math.exp(min(v, 20)):.2f}{' ⭐' if best else ''}", flush=True)
            last_eval = time.time()
        if done or time.time() - last_save >= SAVE_EVERY_MIN * 60:
            st['elapsed'] = elapsed(); st['complete'] = done
            save_ckpt(name, params=params, opt_state=opt_state, bias=bias, state=st)
            last_save = time.time()
        if done:
            break
    return params, bias, st

def init_pretrain():
    return init_params(KEY), jnp.zeros((max(1, N_MOE), N_EXPERTS), F32)

PARAMS, BIAS, PT_STATE = run_stage('pretrain', PRETRAIN_MINUTES, make_tx(1.0), init_pretrain,
                                   lambda rng, mi, ac: pretrain_batch(rng, TRAIN_TOK, mi, ac),
                                   MICRO_BATCH, ACCUM, f'Muon lr={MUON_LR}, Adam lr={ADAM_LR}', WARMUP_STEPS)
print(f"✅ Pretrain: {PT_STATE['step']} qadam, {PT_STATE['tokens'] / 1e9:.3f}B token, eng yaxshi val={PT_STATE['best_val']:.4f}")

## 6. SFT (chat)
Pretrain vaznlaridan boshlanadi, LR ×0.25, yangi optimizer holati. Format: `<|bos|><|user|>\n{savol}\n<|assistant|>\n{javob}<|eos|>`; loss faqat javob tokenlarida, misollar qatorga joylanadi (packing).

In [ ]:
# 5. SFT: o'zbekcha chat (uzbek-lang-lm) + kod ko'rsatmalari; loss faqat javoblarda
if SFT_MINUTES > 0 and len(SFT['train']['idx']) and len(SFT['val']['idx']):
    _pt_params, _pt_bias = PARAMS, BIAS
    SFT_PARAMS, SFT_BIAS, SFT_STATE = run_stage(
        'sft', SFT_MINUTES, make_tx(SFT_LR_SCALE), lambda: (jax.device_get(_pt_params), jax.device_get(_pt_bias)),
        lambda rng, mi, ac: sft_batch(rng, SFT['train'], mi, ac), SFT_MICRO_BATCH, SFT_ACCUM,
        f'LR x{SFT_LR_SCALE} (pretraindan)', 20)
    print(f"✅ SFT: {SFT_STATE['step']} qadam, eng yaxshi val={SFT_STATE['best_val']:.4f}")
else:
    SFT_PARAMS, SFT_BIAS = PARAMS, BIAS
    print('SFT o\'tkazib yuborildi (SFT_MINUTES=0 yoki chat data yo\'q).')

## 7. Sinov: matn yaratish va chat
`ask("savol")` bilan istalgan savolni bering. `temperature`, `top_k`, `top_p`, `rep_penalty` sozlanadi.

In [ ]:
# 6. Matn yaratish va chat (MoE inference'da dropless — hech bir token tashlanmaydi)
@jax.jit
def next_logits(params, bias, buf, pos):
    x, _, _ = forward(params, bias, buf, None)
    h = jax.lax.dynamic_index_in_dim(x[0], pos - 1, axis=0, keepdims=False)
    return head_logits(params, rms(h, params['norm_f']))

def generate(params, bias, ids, max_new=200, temperature=0.7, top_k=50, top_p=0.95, rep_penalty=1.15, seed=0,
             stop=(tok.eos_id, tok.user_id)):
    rng, ids, out = np.random.default_rng(seed), list(ids), []
    for _ in range(max_new):
        ctx = ids[-(SEQ - 1):]
        buf = np.full((1, SEQ), PAD, np.int32); buf[0, :len(ctx)] = ctx
        lg = np.asarray(next_logits(params, bias, buf, len(ctx)), np.float64)
        if rep_penalty != 1.0:                       # takrorlanishni kamaytirish (kichik modellar uchun muhim)
            seen = np.unique(ctx[-256:]); lg[seen] = np.where(lg[seen] > 0, lg[seen] / rep_penalty, lg[seen] * rep_penalty)
        lg[[PAD, BOS]] = -np.inf
        if temperature <= 0:
            nxt = int(lg.argmax())
        else:
            lg = lg / temperature
            if top_k: lg[lg < np.sort(lg)[-top_k]] = -np.inf
            p = np.exp(lg - lg.max()); p /= p.sum()
            order = np.argsort(-p); cum = np.cumsum(p[order])
            p[order[cum - p[order] > top_p]] = 0; p /= p.sum()
            nxt = int(rng.choice(len(p), p=p))
        if nxt in stop:
            break
        ids.append(nxt); out.append(nxt)
    return tok.decode(out)

def ask(question, **kw):
    """Chat: print(ask("O'zbekistonning poytaxti qaysi shahar?"))"""
    return generate(SFT_PARAMS, SFT_BIAS, chat_ids(tok, [('user', question)], add_generation_prompt=True)[0], **kw).strip()

print('=== PRETRAIN modeli: matn davomi ===')
for s in ["O'zbekiston Respublikasi", "Alisher Navoiy", "def fibonacci(n):"]:
    print(f'\n▶ {s}{generate(PARAMS, BIAS, [BOS] + tok.encode(s), max_new=80 if PROFILE == "real" else 20)}')
print('\n=== SFT modeli: chat ===')
for q in ["O'zbekistonning poytaxti qaysi shahar?", "Sog'lom turmush tarzi uchun uchta maslahat bering.",
          "Python'da ro'yxatdagi juft sonlar yig'indisini qaytaruvchi funksiya yozing."][: 3 if PROFILE == 'real' else 1]:
    print(f"\n❓ {q}\n💬 {ask(q, max_new=200 if PROFILE == 'real' else 20)}")

## 8. Saqlash — `.nit` format
Barcha fayllar o'zimizning **`.nit`** formatida (`.pt`/pickle emas):
- `WORKDIR/checkpoints/pretrain.nit`, `sft.nit` — trening holati (vaznlar + optimizer + MoE bias + progress), Colab uzilsa shundan davom etiladi.
- `WORKDIR/export/nanogpt_uz.nit` — **bitta faylda tayyor model**: bf16 vaznlar + o'zbekcha tokenizer + sozlamalar + chat formati.

Format: `NIT1` belgisi → JSON sarlavha (meta + tensorlar ro'yxati) → 64 baytga tekislangan xom tensorlar. Pickle yo'q — `.nit` faylni ochish hech qanday kod bajarmaydi (safetensors kabi xavfsiz). Yuklash: `params, bias, tok, meta = load_nit_model('.../nanogpt_uz.nit')`.

**Keyingi qadamlar:** natija yoqsa — `PRETRAIN_MINUTES` ni oshiring, `D_MODEL`/`N_LAYER` ni kattalashtiring yoki `MTP_WEIGHT=0.1` bilan tajriba qiling. Har o'zgarishni val loss bilan solishtiring (nanoGPT ruhi: o'lchab ko'r).

In [ ]:
# 7. Yakuniy model: BITTA .nit fayl (vaznlar bf16 + tokenizer + sozlamalar) va qayta yuklash tekshiruvi
EXPORT = WORK / 'export'; EXPORT.mkdir(exist_ok=True)
MODEL_FILE = EXPORT / 'nanogpt_uz.nit'
nit_save(MODEL_FILE,
         {k: np.asarray(jax.device_get(v), np.float32).astype(jnp.bfloat16) for k, v in tree_to_named('params', SFT_PARAMS).items()}
         | {'bias': SFT_BIAS},
         {'kind': 'model', 'spec': MODEL_SPEC, 'softcap': SOFTCAP, 'rope_theta': ROPE_THETA, 'tokenizer': tok.tk.to_str(),
          'chat_format': '<|bos|><|user|>\\n{savol}\\n<|assistant|>\\n{javob}<|eos|>', 'total_params': TOTAL_P,
          'active_params': ACTIVE_P, 'pretrain_steps': PT_STATE['step'], 'pretrain_tokens': PT_STATE['tokens']})
print(f"✅ Model: {MODEL_FILE} ({MODEL_FILE.stat().st_size / 2**20:.0f} MiB) — vaznlar, tokenizer va sozlamalar bitta faylda")

def load_nit_model(path):
    """.nit modelni yuklash: params, moe_bias, tokenizer, meta. (Model sozlamalari shu notebookdagidek bo'lishi kerak.)"""
    meta, t = nit_load(path)
    if meta.get('kind') != 'model' or meta['spec'] != MODEL_SPEC:
        raise ValueError(f'Sozlamalar mos emas: {meta.get("spec")}')
    params = named_to_tree('params', {k: jnp.asarray(v, F32) for k, v in t.items() if k.startswith('params')},
                           jax.eval_shape(init_params, KEY))
    return params, jnp.asarray(t['bias'], F32), UzTokenizer(Tokenizer.from_str(meta['tokenizer'])), meta

# Tekshiruv: .nit dan qayta yuklangan model asl model bilan bir xil javob beradi
_p, _b, _tok, _meta = load_nit_model(MODEL_FILE)
_ids = chat_ids(_tok, [('user', "O'zbekiston haqida qisqacha ma'lumot bering.")], add_generation_prompt=True)[0]
_buf = np.full((1, SEQ), PAD, np.int32); _buf[0, :len(_ids)] = _ids
_d = float(jnp.abs(next_logits(_p, _b, _buf, len(_ids)) - next_logits(SFT_PARAMS, SFT_BIAS, _buf, len(_ids))).max())
print(f'Qayta yuklash tekshiruvi: tokenizer mos={_tok.encode("salom dunyo") == tok.encode("salom dunyo")}, '
      f'logit farqi (bf16 yaxlitlash) = {_d:.3g}')
del _p, _b
hist = PT_STATE.get('history', [])
if hist:
    print('Pretrain val loss tarixi (qadam, daqiqa, loss):', hist[-6:])
print(f"Jami vaqt: {(time.time() - T0) / 60:.1f} daqiqa | Model: {TOTAL_P / 1e6:.0f}M jami / {ACTIVE_P / 1e6:.0f}M faol")